# Conditional V018 final fit: confirmed weighted median plus training pressure grid

Prepared prospectively; no E or secondary outcome has been read. Default execution is inert. This stage requires both independently audited confirmations to pass and exact expected result/audit/source hashes. It does not submit automatically.

Reuse the accepted0.18025 all-approved-data compact/deep/curve/memory/offset checkpoints and their test predictions only after exact manifest, data, recipe, model, training budget, preprocessing, source and row checks. Train only the new deep curve-loss model on all67,905 approved breaths for its A-selected196epoch/rate prefix. Reuse the reviewed V011 fixed-fit/inference machinery with the identical V5PressureRNN constructor; it records the new gate/source identity. No E-based tuning. Refit the pressure codebook on all approved labeled inspiratory training rows. Final inspired predictions use the fixed weighted median midpoint and nearest training grid; unscored phase rows use the fixed mean. Preserve all603600IDs, finite native units, old submissions and exact checkpoint resume.

The new output directory is `outputs/confirmation_v014/final_all_v018`. Final refitting has no independent holdout. Discussion-inspired source provenance is recorded separately. This notebook is prepared, not evidence that either confirmation or final fitting completed.


In [ ]:
import os,json,hashlib,gc,time,fcntl,copy
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import pandas as pd
import torch
V018_SOURCE_DEPENDENCIES={"controlled_experiments_v013.ipynb":"56cb9d7e824e58126bddd71c7c0f7063844974c31ee00a6dac23f2112657615a","final_fit_v011.ipynb":"35abc3371ddaf717ca3fc7ff908f4e06517e642c0d0fc6b089342870f533c6de","research/postprocessing_components_v016.ipynb":"4ee67389039846557d74cc1c55884c9a99f0e571745af4febefa4b5c3bce6a0b"}
V018_ACCEPTED_MANIFEST_SHA="3746753aa6cd7eaa306f2279e24d5a5231b6bae9c700bcd703e54b756a823aaf"
V018_ACCEPTED_CSV_SHA="c344a485f5ef1290aadf5dfd379333009efab674b7ccdff6f361e225a4b8b77a"
V018_NOTEBOOK="final_fit_v018.ipynb"

def v018_require(c,m):
    if not bool(c):raise ValueError(m)
def v018_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def v018_code(p):return "\n".join("".join(c["source"]) for c in json.loads(Path(p).read_text())["cells"] if c["cell_type"]=="code")
def v018_path(root,value):
    marker="/ventilator-pressure-prediction/";value=str(value);p=(root/(value.split(marker,1)[1] if marker in value else value)).resolve()
    v018_require(p.is_relative_to(root),"Path outside project");return p

def v018_primary_secondary_gate(root,expected_primary_sha,expected_primary_audit_sha,expected_secondary_sha,expected_secondary_audit_sha):
    E=root/"outputs/confirmation_v014";S=root/"outputs/postprocessing_confirmation_v017"
    paths=[E/"confirmation_result.json",root/"research/confirmation_review_v014.json",S/"confirmation_result.json",root/"research/postprocessing_confirmation_review_v017.json"]
    expected=[expected_primary_sha,expected_primary_audit_sha,expected_secondary_sha,expected_secondary_audit_sha]
    for p,h in zip(paths,expected):v018_require(isinstance(h,str) and len(h)==64 and v018_sha(p)==h,"Missing/changed reviewed final prerequisite: "+str(p))
    primary,pa,secondary,sa=[json.loads(p.read_text()) for p in paths]
    v018_require(primary["status"]=="complete" and primary["passed"] and secondary["status"]=="complete" and secondary["passed"],"Both fixed confirmation stages must pass")
    v018_require(pa["status"]==sa["status"]=="passed" and pa["confirmation_result_sha256"]==expected_primary_sha and sa["confirmation_result_sha256"]==expected_secondary_sha,"Independent confirmation audits do not match")
    v018_require(secondary["primary_result_sha256"]==expected_primary_sha and secondary["primary_audit_sha256"]==expected_primary_audit_sha and secondary["selection_sha256"]==primary["selection_sha256"]==pa["selection_sha256"],"Confirmation selection identity differs")
    v018_require(secondary["timing_gate_passed"] and secondary["protocol_sha256"]=="97214e460ea200a836922245363132c09ef54b133080b89bb342c4e1d191d627" and secondary["source_code_sha256"]=="d11073d1ded4945ef5d6fe60da1d0b1ef5ee0240e0990ea8dfe1fc891ab4abcd" and primary["source_code_sha256"]=="0f22f8168da6e9304c541ab8415c573d6ad4efdf31d5a1391619448c8d478719","Preregistered source/timing changed")
    v018_require(all(c["gain"]>0 and c["high"]<0 for c in secondary["comparisons"].values()),"Secondary fixed comparisons failed")
    protocol_path=root/"research/confirmation_protocol_v014.json"
    v018_require(v018_sha(protocol_path)=="dee5840ee16262dc282ebeb98643cb24e312e55761634f8fae074ed6c921f6f1","Frozen E recipes changed")
    plan=json.loads(protocol_path.read_text())
    expected_weights=dict(original_compact=.125,curve=.125,memory=.125,offset=.125,deep=.25,deep_curve=.25)
    v018_require(primary["winner_weights"]==plan["winner_weights"]==expected_weights and secondary["policy"]=="weighted_median_midpoint_then_nearest_training_pressure_codebook","Selected final recipe changed")
    return plan,dict(primary_result_sha256=expected_primary_sha,primary_audit_sha256=expected_primary_audit_sha,secondary_result_sha256=expected_secondary_sha,secondary_audit_sha256=expected_secondary_audit_sha,selection_sha256=primary["selection_sha256"],postprocessing_protocol_sha256=secondary["protocol_sha256"])

def run_final_v018(root=".",resume=False,expected_primary_sha256=None,expected_primary_audit_sha256=None,expected_secondary_sha256=None,expected_secondary_audit_sha256=None,expected_source_code_sha256=None):
    root=Path(root).resolve();out=root/"outputs/confirmation_v014/final_all_v018";locks=[];began=time.monotonic();previous_cwd=Path.cwd()
    try:
        os.chdir(root)
        lock=(root/"outputs/.v011_gpu.lock").open('a+');fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB);locks.append(lock)
        v018_require(torch.cuda.is_available(),"Approved actual CUDA required")
        for flag in ["PA3_RUN_V013","PA3_RUN_FINAL_V011"]:v018_require(os.environ.get(flag,"0")!="1","Other notebook run flag enabled")
        current=hashlib.sha256(v018_code(root/V018_NOTEBOOK).encode()).hexdigest();v018_require(current==expected_source_code_sha256,"Final source differs from reviewed pin")
        plan,gate=v018_primary_secondary_gate(root,expected_primary_sha256,expected_primary_audit_sha256,expected_secondary_sha256,expected_secondary_audit_sha256)
        gate["final_source_code_sha256"]=current
        for name,h in V018_SOURCE_DEPENDENCIES.items():
            code=v018_code(root/name);v018_require(hashlib.sha256(code.encode()).hexdigest()==h,"Reusable source changed: "+name);exec(compile(code,name,"exec"),globals())
        old_lock=json.loads((root/"outputs/confirmation_v011/locked_before_D.json").read_text());old_split=json.loads((root/"outputs/confirmation_v011/D_split.json").read_text())
        v011_load_components(root,old_lock["source_fingerprints"]);torch.set_num_threads(4);seed_sequence(20261102)
        old_path=root/"outputs/confirmation_v011/final_all/final_manifest.json";v018_require(v018_sha(old_path)==V018_ACCEPTED_MANIFEST_SHA,"Accepted five-model manifest changed")
        accepted=json.loads(old_path.read_text());v018_require(v018_sha(v018_path(root,accepted["submission_file"]))==V018_ACCEPTED_CSV_SHA,"Accepted official CSV changed")
        if (out/"final_manifest.json").exists():
            complete=json.loads((out/"final_manifest.json").read_text())
            v018_require(resume and complete["status"]=="complete" and complete["gate"]==gate and complete["weights"]==plan["winner_weights"],"Completed final manifest changed or missing explicit resume")
            for name,h in [("train.csv",V011_TRAIN_SHA),("test.csv",V011_TEST_SHA),("sample_submission.csv",V011_SAMPLE_SHA)]:v018_require(v018_sha(root/"data"/name)==h,"Approved data changed")
            for record in complete["members"].values():v018_require(v018_sha(v018_path(root,record["checkpoint_path"]))==record["checkpoint_sha256"],"Completed final checkpoint changed")
            for path,h in [(v018_path(root,complete["submission"]["file"]),complete["submission_sha256"]),(out/"final_test_predictions.npy",complete["prediction_file_sha256"]),(root/complete["grid_file"],complete["grid_file_sha256"])]:v018_require(v018_sha(path)==h,"Completed final artifact changed")
            return complete
        data=v011_load_all_training(root,old_lock,old_split)
        v018_require(data["data_hashes"]==accepted["all_training_data_fingerprints"] and data["group_sha256"]==accepted["all_training_group_sha256"] and _v8_statistics_sha256(data["prep"])==accepted["all_training_preprocessing_sha256"],"Verified all-training data/statistics changed")
        out.mkdir(parents=True,exist_ok=True)
        sources=dict(old_lock["source_fingerprints"],**V018_SOURCE_DEPENDENCIES,**{V018_NOTEBOOK:current,"primary_E_result":expected_primary_sha256,"secondary_E_result":expected_secondary_sha256,"confirmation_protocol_v014.json":"dee5840ee16262dc282ebeb98643cb24e312e55761634f8fae074ed6c921f6f1"})
        design=dict(schema="v018-confirmed-fixed-postprocessing-final-v1",gate=gate,weights=plan["winner_weights"],new_member_recipe=plan["members"]["deep_curve"],all_training_breaths=67905,scored_training_points=2061922,accepted_manifest_sha256=V018_ACCEPTED_MANIFEST_SHA,accepted_CSV_sha256=V018_ACCEPTED_CSV_SHA,source_fingerprints=sources,aggregation="fixed weighted median midpoint then nearest all-training inspiratory grid",unscored_phase_rule="fixed six-member mean",seed=20261102,final_fit_has_independent_holdout=False)
        v011_immutable_json(out/"design.json",design,resume)
        reused={};prior_path=root/V011_PRIOR_DIR/"final_manifest.json";v018_require(v018_sha(prior_path)==V011_PRIOR_MANIFEST_SHA,"Original accepted provenance changed");prior=json.loads(prior_path.read_text());v011_verify_legacy_lock(root,prior)
        for name,record in accepted["members"].items():
            recipe=plan["members"][name];v018_require(record["recipe"]==recipe,"Accepted model recipe no longer matches selected recipe")
            checkpoint=v018_path(root,record["checkpoint_path"]);v018_require(v018_sha(checkpoint)==record["checkpoint_sha256"],"Accepted checkpoint changed")
            if name in ["deep","original_compact"]:
                verified=v011_reuse_original(root,dict(member_id=name,recipe=recipe),data,prior)
            else:
                rp=v018_path(root,record["report_file"]);v018_require(v018_sha(rp)==record["report_sha256"],"Accepted report changed")
                report=json.loads(rp.read_text());v018_require(report["completed"] and report["training_breaths"]==67905 and not report["validation_used"] and report["executed_epochs"]==recipe["selected_epochs"] and report["executed_lr_schedule"]==recipe["lr_schedule"],"Accepted all-training budget incomplete")
                model,prep,payload=load_checkpoint_v008(checkpoint,device="cpu",model_factory=v011_factory(name),expected_arm=record["arm"],expected_loss_kind=recipe["loss_kind"],source_fingerprints=accepted["source_fingerprints"])
                v011_check_fixed_payload(payload,dict(recipe=recipe),data["data_hashes"],data["prep"]);del model,prep,payload
                verified=record
            reused[name]=dict(verified,reuse_controller="v018",accepted_manifest_sha256=V018_ACCEPTED_MANIFEST_SHA);gc.collect()
        # Check every reused inference artifact before any new optimizer update.
        test_ids=pd.read_csv(root/"data/test.csv",usecols=["id"],dtype=dict(id="int64")).id.to_numpy()
        sample_ids=pd.read_csv(root/"data/sample_submission.csv",usecols=["id"],dtype=dict(id="int64")).id.to_numpy()
        v018_require(len(test_ids)==603600 and len(np.unique(test_ids))==603600 and np.array_equal(test_ids,sample_ids),"Approved test/sample IDs changed")
        v018_require(v018_sha(v018_path(root,accepted["test_row_ids_file"]))==accepted["test_row_ids_file_sha256"] and np.array_equal(test_ids,np.load(v018_path(root,accepted["test_row_ids_file"]))),"Accepted test rows changed")
        for name,record in reused.items():
            meta=accepted["prediction_members"][name];path=v018_path(root,meta["file"]);v018_require(v018_sha(path)==meta["file_sha256"],"Accepted inference cache changed before training")
            a=np.load(path,allow_pickle=False)
            v018_require(a.dtype==np.float64 and a.shape==(603600,) and np.isfinite(a).all() and v011_array_sha(a)==meta["array_sha256"] and meta["row_ids_sha256"]==v011_array_sha(test_ids) and meta["checkpoint_sha256"]==record["checkpoint_sha256"] and meta["test_sha256"]==V011_TEST_SHA,"Accepted inference cache identity changed before training")
        v018_require(all(len(h)==64 and all(c in "0123456789abcdef" for c in h) for h in sources.values()),"Source fingerprint is not a SHA256 digest")
        grid=pressure_codebook_v016(data["y"],data["out"])
        member=dict(member_id="deep_curve",factory_id="curve",recipe=copy.deepcopy(plan["members"]["deep_curve"]))
        # The source-pinned curve factory is the same V5PressureRNN constructor
        # used by the validated deep curve model; dimensions/seed/loss are unchanged.
        new_record=v011_fit_new_member(root,out,member,data,gate,sources,resume)
        training_meta=dict(group_sha256=data["group_sha256"],data_fingerprints=data["data_hashes"],preprocessing_sha256=_v8_statistics_sha256(data["prep"]))
        del data;gc.collect();torch.cuda.empty_cache()
        ids,groups,new_predictions,new_prediction_meta=v011_infer_members(root,out,[member],dict(deep_curve=new_record),sources,resume)
        predictions=dict(new_predictions);prediction_meta=dict(new_prediction_meta)
        for name,record in reused.items():
            meta=accepted["prediction_members"][name];p=v018_path(root,meta["file"]);v018_require(v018_sha(p)==meta["file_sha256"],"Accepted cached test predictions changed")
            a=np.load(p,allow_pickle=False)
            v018_require(a.dtype==np.float64 and a.shape==ids.shape and np.isfinite(a).all() and v011_array_sha(a)==meta["array_sha256"] and meta["row_ids_sha256"]==v011_array_sha(ids) and meta["checkpoint_sha256"]==record["checkpoint_sha256"] and meta["test_sha256"]==V011_TEST_SHA,"Accepted test cache identity changed")
            predictions[name]=a;prediction_meta[name]=meta
        weights=plan["winner_weights"];base=sum(w*predictions[k] for k,w in weights.items());processed=selected_postprocessing_v016(predictions,weights,grid)
        phase=pd.read_csv(root/"data/test.csv",usecols=["id","u_out"],dtype=dict(id="int64",u_out="int8"))
        v018_require(np.array_equal(phase.id.to_numpy(),ids),"Test phase/ID alignment changed")
        final=np.where(phase.u_out.to_numpy()==0,processed,base);v018_require(np.isfinite(final).all() and final.shape==(603600,),"Final predictions invalid")
        for path,value in [(out/"all_training_inspiratory_pressure_grid.npy",grid),(out/"test_row_ids.npy",ids),(out/"final_test_predictions.npy",final)]:
            if path.exists():v018_require(resume and np.array_equal(np.load(path),value),"Existing final array changed")
            else:v011_save_array(path,value)
        submission=v011_write_submission(out,ids,final,resume)
        v018_require(v018_sha(v018_path(root,accepted["submission_file"]))==V018_ACCEPTED_CSV_SHA,"Best official CSV changed during work")
        _,after=v018_primary_secondary_gate(root,expected_primary_sha256,expected_primary_audit_sha256,expected_secondary_sha256,expected_secondary_audit_sha256);v018_require(dict(after,final_source_code_sha256=current)==gate,"Final prerequisite changed during work")
        manifest=dict(status="complete",schema=design["schema"],gate=gate,design_sha256=v011_json_sha(design),members=dict(reused,deep_curve=new_record),prediction_members=prediction_meta,weights=weights,aggregation=design["aggregation"],unscored_phase_rule=design["unscored_phase_rule"],source_fingerprints=sources,training=training_meta,all_training_breaths=67905,scored_training_points=2061922,grid_file=str((out/"all_training_inspiratory_pressure_grid.npy").relative_to(root)),grid_file_sha256=v018_sha(out/"all_training_inspiratory_pressure_grid.npy"),grid_levels=len(grid),prediction_file_sha256=v018_sha(out/"final_test_predictions.npy"),prediction_array_sha256=v011_array_sha(final),row_ids_array_sha256=v011_array_sha(ids),submission=submission,submission_sha256=submission["file_sha256"],old_CSV_sha256=V018_ACCEPTED_CSV_SHA,final_fit_has_independent_holdout=False,elapsed_seconds=time.monotonic()-began,submitted=False,completed_utc=datetime.now(timezone.utc).isoformat())
        v011_write_json(out/"final_manifest.json",manifest);return manifest
    finally:
        os.chdir(previous_cwd)
        for f in reversed(locks):fcntl.flock(f,fcntl.LOCK_UN);f.close()

if os.environ.get("PA3_RUN_FINAL_V018","0")=="1":
    V018_RESULT=run_final_v018(Path.cwd(),resume=os.environ.get("PA3_RESUME_FINAL_V018","0")=="1",expected_primary_sha256=os.environ.get("PA3_V018_PRIMARY_SHA256"),expected_primary_audit_sha256=os.environ.get("PA3_V018_PRIMARY_AUDIT_SHA256"),expected_secondary_sha256=os.environ.get("PA3_V018_SECONDARY_SHA256"),expected_secondary_audit_sha256=os.environ.get("PA3_V018_SECONDARY_AUDIT_SHA256"),expected_source_code_sha256=os.environ.get("PA3_V018_SOURCE_SHA256"))
    print(json.dumps(V018_RESULT["submission"],indent=2))
